# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** 05

**Team members:**
- Financial Services Pipeline Lead: Carmen Li
- Travel and Hospitality Pipeline Lead: Sardorbek Shorobov
- Comparison and Visualization Lead (Integrator): Lionel Nash

**Submission filename:** MP03_Notebook_team_05.ipynb

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_<NN>.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [1]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet

In [2]:
import json
import re
import time
import os
import html
from datetime import date, datetime, timedelta
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholders below before running/submitting.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
TEAM_NUMBER = "05"  # replace with your two-digit Brightspace team number
USER_AGENT = "CIS3120 MP03 Team 05 - sardorbek.shorobov@baruchmail.cuny.edu"  # replace <NN> and email

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

# Token-price constants from the MP03 assignment.
INPUT_TOKEN_PRICE_PER_MILLION = 1.00
OUTPUT_TOKEN_PRICE_PER_MILLION = 5.00


In [3]:
# Configure the Anthropic API client from Colab Secrets.
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [4]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, use the assignment seed lists.
import sys

# When running in Colab from a cloned repo, this places the repo root on sys.path.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from mp03.seeds import (
        FINANCIAL_SERVICES_TICKERS,
        FINANCIAL_SERVICES_PHRASES,
        TRAVEL_HOSPITALITY_TICKERS,
        TRAVEL_HOSPITALITY_PHRASES,
    )

    FINANCIAL_SERVICES_TICKERS = [
        # Money-center banks (seed)
        "JPM", "BAC", "WFC", "C",
        # Regional banks (seed)
        "PNC", "USB", "TFC",
        # Asset management (seed)
        "BLK", "BX",
        # Insurance (seed)
        "MET", "PRU",
        # Payments (seed)
        "V", "MA", "AXP",
        # Capital markets — added
        "GS", "MS", "SCHW",
        # Fintech — added
        "PYPL", "SQ",
    ]

    # Company-name embedded phrases — guarantees EDGAR returns these companies' filings
    # This is the key fix: instead of searching generically and filtering by ticker,
    # we search for company name + location keyword together.
    FINANCIAL_SERVICES_PHRASES = [
        # Seed phrases (kept for broad recall)
        '"new branch"',
        '"branch opening"',
        '"branch closure"',
        '"branch closing"',
        '"branch consolidation"',
        '"regional office"',
        '"office closure"',
        '"operations center"',
        '"data center"',
        '"new location"',
        # Extended phrases
        '"new office"',
        '"office opening"',
        '"trading floor"',
        '"advisory office"',
    ]

    # Company-name targeted phrases for direct EDGAR hits
    # These are searched separately and merged with the above
    FINANCIAL_SERVICES_COMPANY_PHRASES = [
        '"JPMorgan" "branch"',
        '"Wells Fargo" "branch"',
        '"Bank of America" "branch"',
        '"Citibank" "branch"',
        '"Truist" "branch"',
        '"PNC" "branch"',
        '"U.S. Bank" "branch"',
        '"Goldman Sachs" "office"',
        '"Morgan Stanley" "office"',
        '"Charles Schwab" "branch"',
        '"American Express" "office"',
        '"Mastercard" "office"',
        '"MetLife" "office"',
        '"Prudential" "office"',
        '"BlackRock" "office"',
        '"Blackstone" "office"',
    ]

    # Combined list used by the pipeline
    ALL_FS_PHRASES = FINANCIAL_SERVICES_PHRASES + FINANCIAL_SERVICES_COMPANY_PHRASES

except ModuleNotFoundError:
    # Fallback seed lists copied from the MP03 assignment so the notebook can
    # still run if opened directly in Colab without the full repo package.
    FINANCIAL_SERVICES_TICKERS = [
        # Money-center banks (seed)
        "JPM", "BAC", "WFC", "C",
        # Regional banks (seed)
        "PNC", "USB", "TFC",
        # Asset management (seed)
        "BLK", "BX",
        # Insurance (seed)
        "MET", "PRU",
        # Payments (seed)
        "V", "MA", "AXP",
        # Capital markets — added
        "GS", "MS", "SCHW",
        # Fintech — added
        "PYPL", "SQ",
    ]

    # Company-name embedded phrases — guarantees EDGAR returns these companies' filings
    # This is the key fix: instead of searching generically and filtering by ticker,
    # we search for company name + location keyword together.
    FINANCIAL_SERVICES_PHRASES = [
        # Seed phrases (kept for broad recall)
        '"new branch"',
        '"branch opening"',
        '"branch closure"',
        '"branch closing"',
        '"branch consolidation"',
        '"regional office"',
        '"office closure"',
        '"operations center"',
        '"data center"',
        '"new location"',
        # Extended phrases
        '"new office"',
        '"office opening"',
        '"trading floor"',
        '"advisory office"',
    ]

    # Company-name targeted phrases for direct EDGAR hits
    # These are searched separately and merged with the above
    FINANCIAL_SERVICES_COMPANY_PHRASES = [
        '"JPMorgan" "branch"',
        '"Wells Fargo" "branch"',
        '"Bank of America" "branch"',
        '"Citibank" "branch"',
        '"Truist" "branch"',
        '"PNC" "branch"',
        '"U.S. Bank" "branch"',
        '"Goldman Sachs" "office"',
        '"Morgan Stanley" "office"',
        '"Charles Schwab" "branch"',
        '"American Express" "office"',
        '"Mastercard" "office"',
        '"MetLife" "office"',
        '"Prudential" "office"',
        '"BlackRock" "office"',
        '"Blackstone" "office"',
    ]

    # Combined list used by the pipeline
    ALL_FS_PHRASES = FINANCIAL_SERVICES_PHRASES + FINANCIAL_SERVICES_COMPANY_PHRASES

    TRAVEL_HOSPITALITY_TICKERS = [
        "MAR", "HLT", "H", "CHH", "WH",
        "CCL", "RCL", "NCLH",
        "DAL", "UAL", "AAL", "LUV",
        "BKNG", "EXPE",
    ]

    TRAVEL_HOSPITALITY_PHRASES = [
        '"new property"',
        '"new hotel"',
        '"hotel opening"',
        '"resort opening"',
        '"property opening"',
        '"brand conversion"',
        '"new route"',
        '"new gateway"',
        '"new terminal"',
        '"grand opening"',
    ]

# Industry 2 additions for Travel and Hospitality.
# The seed list is kept, then expanded to lodging REITs, resort/casino operators,
# additional airlines, and vacation ownership firms. These are still hospitality/travel
# companies and are useful because EDGAR location-event hits often come from property
# owners/operators, not only brand managers like Hilton or Marriott.
TRAVEL_HOSPITALITY_TICKERS_EXTENDED = list(dict.fromkeys(
    TRAVEL_HOSPITALITY_TICKERS + [
        # Hotel REITs / lodging property companies
        "BHR", "HST", "PK", "APLE", "SHO", "PEB", "RLJ", "XHR", "DRH", "INN",
        # Resort / casino hospitality operators
        "MGM", "LVS", "WYNN", "CZR", "BYD",
        # Additional airlines / travel operators
        "ALK", "JBLU", "SAVE",
        # Vacation ownership / timeshare hospitality
        "VAC", "TNL", "HGV",
    ]
))

# Phrase extensions: include both exact phrases and a few safer broad terms.
# Exact quoted phrases improve precision; unquoted terms help recall because SEC full-text
# search can be inconsistent with long quoted phrases.
TRAVEL_HOSPITALITY_PHRASE_EXTENSIONS = [
    '"new resort"',
    '"resort expansion"',
    '"new destination"',
    '"route expansion"',
    '"hotel conversion"',
    '"property conversion"',
    '"airport lounge"',
    'hotel opening',
    'new hotel',
    'new resort',
    'new route',
    'route expansion',
    'brand conversion',
    'property opening',
    'resort expansion',
    'hotel',
]

TRAVEL_HOSPITALITY_PHRASES_FINAL = list(dict.fromkeys(
    TRAVEL_HOSPITALITY_PHRASES + TRAVEL_HOSPITALITY_PHRASE_EXTENSIONS
))

print(f"Tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Generic phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Company-targeted phrases: {len(FINANCIAL_SERVICES_COMPANY_PHRASES)}")
print(f"Total phrases: {len(ALL_FS_PHRASES)}")

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality seed tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality extended tickers: {len(TRAVEL_HOSPITALITY_TICKERS_EXTENDED)}")
print(f"Travel and Hospitality final phrases: {len(TRAVEL_HOSPITALITY_PHRASES_FINAL)}")

Tickers: 19
Generic phrases: 14
Company-targeted phrases: 16
Total phrases: 30
Financial Services tickers: 19
Financial Services phrases: 14
Travel and Hospitality seed tickers: 14
Travel and Hospitality extended tickers: 35
Travel and Hospitality final phrases: 26


---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [5]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,

) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [6]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped


### Stage 2 — Fetch the press release text from each filing

In [7]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [8]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [9]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [10]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [11]:
# Optional diagnostics switch.
# Keep False for final run/submission so the notebook has no leftover debug output.
RUN_TRAVEL_DIAGNOSTICS = False

In [12]:
# Shared run metadata for the window-results table.
RUN_METADATA: dict[tuple[str, int], dict] = {}
_TICKER_CIK_CACHE: dict[tuple[str, ...], dict[str, str]] = {}


def estimate_claude_cost(input_tokens: int, output_tokens: int) -> float:
    """Estimate Claude cost using the MP03 Haiku 4.5 pricing rule."""
    input_cost = (int(input_tokens) / 1_000_000) * INPUT_TOKEN_PRICE_PER_MILLION
    output_cost = (int(output_tokens) / 1_000_000) * OUTPUT_TOKEN_PRICE_PER_MILLION
    return input_cost + output_cost


def first_nonempty(value, default=None):
    """Return the first non-empty item from a list-like value, or the value itself."""
    if isinstance(value, (list, tuple)):
        for item in value:
            if item not in (None, ""):
                return item
        return default
    if value not in (None, ""):
        return value
    return default


def get_ticker_cik_map(ticker_list: list[str]) -> dict[str, str]:
    """Build a ticker-to-CIK map from the SEC company tickers JSON.

    CIKs are returned without leading zero padding so they match EDGAR search hits.
    Results are cached by ticker list to avoid repeated SEC requests.
    """
    cache_key = tuple(sorted({str(t).upper() for t in ticker_list}))
    if cache_key in _TICKER_CIK_CACHE:
        return _TICKER_CIK_CACHE[cache_key]

    url = "https://www.sec.gov/files/company_tickers.json"
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()

    data = response.json()
    ticker_set = set(cache_key)
    ticker_to_cik: dict[str, str] = {}

    for item in data.values():
        ticker = str(item.get("ticker", "")).upper()
        cik = str(item.get("cik_str", "")).lstrip("0")
        if ticker in ticker_set and cik:
            ticker_to_cik[ticker] = cik

    _TICKER_CIK_CACHE[cache_key] = ticker_to_cik
    return ticker_to_cik


def ticker_for_hit(hit: dict, ticker_list: list[str]) -> str | None:
    """Infer the best ticker for an EDGAR hit using tickers first, then CIKs."""
    source = hit.get("_source", {})
    ticker_set = {str(t).upper() for t in ticker_list}

    hit_tickers = source.get("tickers", []) or []
    for ticker in hit_tickers:
        ticker = str(ticker).upper()
        if ticker in ticker_set:
            return ticker

    ticker_to_cik = get_ticker_cik_map(ticker_list)
    cik_to_ticker = {str(cik).lstrip("0"): ticker for ticker, cik in ticker_to_cik.items()}
    hit_ciks = source.get("ciks", []) or []
    for cik in hit_ciks:
        ticker = cik_to_ticker.get(str(cik).lstrip("0"))
        if ticker:
            return ticker

    return first_nonempty(hit_tickers)


def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict EDGAR search candidates to a list of tickers.

    The MP03 assignment notes that EDGAR hits may include _source['tickers'].
    In practice, the full-text endpoint often exposes CIKs more reliably than
    tickers, so this implementation checks both fields while preserving the
    required function signature.
    """
    ticker_set = {str(t).upper() for t in ticker_list}
    ticker_to_cik = get_ticker_cik_map(ticker_list)
    cik_set = {str(cik).lstrip("0") for cik in ticker_to_cik.values()}

    filtered: list[dict] = []

    for hit in candidates:
        source = hit.get("_source", {})

        hit_tickers = source.get("tickers", []) or []
        hit_ticker_set = {str(t).upper() for t in hit_tickers}

        hit_ciks = source.get("ciks", []) or []
        hit_cik_set = {str(cik).lstrip("0") for cik in hit_ciks}

        if (hit_ticker_set & ticker_set) or (hit_cik_set & cik_set):
            filtered.append(hit)

    return filtered

In [13]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice.

    Returns geocoded events with an "industry" field added to each record.
    Also stores candidate_count, true_event_count, and estimated_cost_usd in
    RUN_METADATA[(industry_label, window_days)] for the window-results table.
    """
    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)

    # Stage 1: broad phrase search, then industry ticker filter.
    candidates = search_edgar_all_phrases(
        phrases=phrase_list,
        start_date=start_date,
        end_date=end_date,
        forms="8-K",
        max_pages=2,
        max_filings=250,
    )
    filtered_candidates = filter_candidates_by_tickers(candidates, ticker_list)

    geocoded_events: list[dict] = []
    true_event_count = 0
    total_input_tokens = 0
    total_output_tokens = 0
    extraction_errors = 0
    geocode_failures = 0

    for hit in filtered_candidates:
        source = hit.get("_source", {})

        try:
            text, url = fetch_exhibit_text(hit)
        except requests.RequestException:
            extraction_errors += 1
            continue

        matched_ticker = ticker_for_hit(hit, ticker_list)
        filing = {
            "text": text,
            "url": url,
            "company_name": first_nonempty(source.get("display_names"), first_nonempty(source.get("company_names"), "Unknown company")),
            "ticker": matched_ticker,
            "filing_date": source.get("file_date") or source.get("filedAt") or source.get("period_ending"),
            "edgar_id": hit.get("_id"),
        }

        try:
            extracted = extract_with_claude(filing)
        except Exception as exc:
            extraction_errors += 1
            # Keep going so one bad filing does not kill the whole window trial.
            continue

        total_input_tokens += int(extracted.get("input_tokens", 0) or 0)
        total_output_tokens += int(extracted.get("output_tokens", 0) or 0)

        if extracted.get("is_location_event") is not True:
            continue

        true_event_count += 1

        city = extracted.get("city")
        state = extracted.get("state")

        try:
            coordinates = geocode_location(city, state)
        except requests.RequestException:
            coordinates = None

        if coordinates is None:
            geocode_failures += 1
            continue

        lat, lon = coordinates
        extracted["lat"] = lat
        extracted["lon"] = lon
        extracted["industry"] = industry_label

        # Normalize names expected by the map popup.
        extracted["filing_url"] = extracted.get("url")
        extracted["event_type"] = extracted.get("event_type") or "other"

        geocoded_events.append(extracted)

    estimated_cost_usd = estimate_claude_cost(total_input_tokens, total_output_tokens)

    RUN_METADATA[(industry_label, window_days)] = {
        "industry": industry_label,
        "window_days": window_days,
        "candidate_count": len(filtered_candidates),
        "event_count": true_event_count,
        "geocoded_event_count": len(geocoded_events),
        "estimated_cost_usd": estimated_cost_usd,
        "input_tokens": total_input_tokens,
        "output_tokens": total_output_tokens,
        "extraction_errors": extraction_errors,
        "geocode_failures": geocode_failures,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
    }

    return geocoded_events

In [14]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict that is directly appendable to the window-experiment
    results table with the exact columns required by the MP03 assignment.
    """
    return {
        "industry": industry_label,
        "window_days": int(window_days),
        "candidate_count": int(candidate_count),
        "event_count": int(event_count),
        "estimated_cost_usd": round(float(estimated_cost_usd), 6),
    }


---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [15]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [24]:
fs_events_by_window = {}

FS_WINDOWS_TO_TRY = [30, 60, 90, 180, 360]
FS_CHOSEN_WINDOW_DAYS = 360

for window_days in FS_WINDOWS_TO_TRY:

    fs_events_for_window = run_industry_pipeline(
        industry_label="Financial Services",
        ticker_list=FINANCIAL_SERVICES_TICKERS,
        phrase_list=ALL_FS_PHRASES,
        window_days=window_days,
    )

    trial_meta = RUN_METADATA[("Financial Services", window_days)]

    trial_row = summarize_window_trial(
        industry_label="Financial Services",
        window_days=window_days,
        candidate_count=trial_meta["candidate_count"],
        event_count=trial_meta["event_count"],
        estimated_cost_usd=trial_meta["estimated_cost_usd"],
    )

    window_results = pd.concat(
        [window_results, pd.DataFrame([trial_row])],
        ignore_index=True,
    )

    fs_events_by_window[window_days] = fs_events_for_window
    FS_CHOSEN_WINDOW_DAYS = window_days

    print(
        f"Financial Services | {window_days} days | "
        f"{trial_meta['candidate_count']} candidates | "
        f"{trial_meta['event_count']} true events | "
        f"{len(fs_events_for_window)} geocoded | "
        f"${trial_meta['estimated_cost_usd']:.6f}"
    )

    if trial_meta["event_count"] >= 8:
        break

fs_events = fs_events_by_window.get(FS_CHOSEN_WINDOW_DAYS, [])



Financial Services | 30 days | 0 candidates | 0 true events | 0 geocoded | $0.000000
Financial Services | 60 days | 0 candidates | 0 true events | 0 geocoded | $0.000000
Financial Services | 90 days | 0 candidates | 0 true events | 0 geocoded | $0.000000
Financial Services | 180 days | 0 candidates | 0 true events | 0 geocoded | $0.000000
Financial Services | 360 days | 1 candidates | 1 true events | 0 geocoded | $0.002938


In [17]:
# Optional Travel/Hospitality Stage 1 diagnostics.
# Turn on only if candidate_count is unexpectedly zero.
if RUN_TRAVEL_DIAGNOSTICS:
    test_start = date.today() - timedelta(days=360)
    test_end = date.today()

    raw_hits = search_edgar_all_phrases(
        TRAVEL_HOSPITALITY_PHRASES_FINAL,
        start_date=test_start,
        end_date=test_end,
        forms="8-K",
        max_pages=2,
        max_filings=100,
    )

    filtered_hits = filter_candidates_by_tickers(
        raw_hits,
        TRAVEL_HOSPITALITY_TICKERS_EXTENDED,
    )

    print("Raw hits:", len(raw_hits))
    print("Filtered hits:", len(filtered_hits))

    for i, hit in enumerate(filtered_hits[:5]):
        source = hit.get("_source", {})
        print("----", i)
        print("display_names:", source.get("display_names"))
        print("tickers:", source.get("tickers"))
        print("ciks:", source.get("ciks"))

In [18]:
# Diagnostic detail cell intentionally left blank for final run.

In [19]:
# Diagnostic detail cell intentionally left blank for final run.

### 4.2 Window trials — Travel and Hospitality

In [20]:
# Travel and Hospitality window trials.
# This is Sardor's Industry 2 section.
# Set to True when ready to run real trials and spend API calls.
RUN_TRAVEL_WINDOW_TRIALS = True

TRAVEL_WINDOWS_TO_TRY = [30, 60, 90, 180, 360]
travel_events_by_window: dict[int, list[dict]] = {}
TRAVEL_CHOSEN_WINDOW_DAYS = None

if RUN_TRAVEL_WINDOW_TRIALS:
    for window_days in TRAVEL_WINDOWS_TO_TRY:
        th_events_for_window = run_industry_pipeline(
            industry_label="Travel and Hospitality",
            ticker_list=TRAVEL_HOSPITALITY_TICKERS_EXTENDED,
            phrase_list=TRAVEL_HOSPITALITY_PHRASES_FINAL,
            window_days=window_days,
        )

        trial_meta = RUN_METADATA[("Travel and Hospitality", window_days)]
        trial_row = summarize_window_trial(
            industry_label="Travel and Hospitality",
            window_days=window_days,
            candidate_count=trial_meta["candidate_count"],
            event_count=trial_meta["event_count"],
            estimated_cost_usd=trial_meta["estimated_cost_usd"],
        )

        window_results = pd.concat(
            [window_results, pd.DataFrame([trial_row])],
            ignore_index=True,
        )

        travel_events_by_window[window_days] = th_events_for_window
        TRAVEL_CHOSEN_WINDOW_DAYS = window_days

        print(
            f"Travel and Hospitality | {window_days} days | "
            f"{trial_meta['candidate_count']} candidates | "
            f"{trial_meta['event_count']} true events | "
            f"{len(th_events_for_window)} geocoded | "
            f"${trial_meta['estimated_cost_usd']:.6f}"
        )

        # Stop for this industry once the assignment target is reached.
        if trial_meta["event_count"] >= 8:
            break

# Use the last/successful Travel run for your section and map preview.
th_events = travel_events_by_window.get(TRAVEL_CHOSEN_WINDOW_DAYS, []) if TRAVEL_CHOSEN_WINDOW_DAYS else []
all_events = fs_events + th_events if "fs_events" in globals() else th_events

window_results

Travel and Hospitality | 30 days | 42 candidates | 8 true events | 7 geocoded | $0.106736


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0.000000
1,Financial Services,60,0,0,0.000000
2,Financial Services,90,0,0,0.000000
3,Financial Services,180,0,0,0.000000
4,Financial Services,360,1,1,0.002943
5,Travel and Hospitality,30,42,8,0.106736


### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [21]:
# Selected window and final pipeline runs.
#
# For the final integrated notebook, the integrator should set CHOSEN_WINDOW_DAYS
# to the smallest window where BOTH industries meet the target, or 360 if one/both
# fall short. As the Travel lead, you can usually leave this as None because the
# Travel window-trial cell above already stores your latest th_events.

CHOSEN_WINDOW_DAYS = None  # replace with 30, 60, 90, 180, or 360 only for final integrated rerun

# Keep teammate/integrator variables safe if their section has not been merged yet.
fs_events = globals().get("fs_events", [])
th_events = globals().get("th_events", [])
all_events = fs_events + th_events

if CHOSEN_WINDOW_DAYS is not None:
    # Financial Services should be run/confirmed by the Financial Services lead.
    # Uncomment only when the team is ready for the final integrated run.
    #
    # fs_events = run_industry_pipeline(
    #     "Financial Services",
    #     FINANCIAL_SERVICES_TICKERS,
    #     ALL_FS_PHRASES,
    #     window_days=CHOSEN_WINDOW_DAYS,
    # )

    th_events = run_industry_pipeline(
        "Travel and Hospitality",
        TRAVEL_HOSPITALITY_TICKERS_EXTENDED,
        TRAVEL_HOSPITALITY_PHRASES_FINAL,
        window_days=CHOSEN_WINDOW_DAYS,
    )

    all_events = fs_events + th_events

print(f"Financial Services:       {len(fs_events)} geocoded events")
print(f"Travel and Hospitality:   {len(th_events)} geocoded events")
print(f"Total:                    {len(all_events)} geocoded events")

# Optional export for the integrator.
output_dir = repo_root / "data"
output_dir.mkdir(exist_ok=True)
if th_events:
    pd.DataFrame(th_events).to_csv(
        output_dir / f"travel_hospitality_events_team_{TEAM_NUMBER}.csv",
        index=False,
    )

Financial Services:       0 geocoded events
Travel and Hospitality:   7 geocoded events
Total:                    7 geocoded events


---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [22]:
# Construct the integrated map.
#
# This cell will render an empty base map if all_events is empty. After the
# final pipelines run, every event marker includes the assignment-required
# popup fields.

industry_icon_colors = {
    "Financial Services": "darkblue",
    "Travel and Hospitality": "cadetblue",
}

event_type_icons = {
    "opening": "plus-circle",
    "closing": "times-circle",
    "relocation": "exchange",
    "expansion": "arrow-up",
    "other": "info-circle",
    None: "info-circle",
}

m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

for event in all_events:
    event_type = event.get("event_type") or "other"
    industry = event.get("industry") or "Unknown"

    popup_html = f"""
    <b>{html.escape(str(event.get("company_name", "Unknown company")))}</b><br>
    <b>Ticker:</b> {html.escape(str(event.get("ticker", "N/A")))}<br>
    <b>Industry:</b> {html.escape(str(industry))}<br>
    <b>Filing date:</b> {html.escape(str(event.get("filing_date", "N/A")))}<br>
    <b>Event type:</b> {html.escape(str(event_type))}<br>
    <b>Summary:</b> {html.escape(str(event.get("summary", "N/A")))}<br>
    <a href="{html.escape(str(event.get("filing_url", event.get("url", "#"))))}" target="_blank">SEC filing</a>
    """

    folium.Marker(
        location=[event["lat"], event["lon"]],
        popup=folium.Popup(popup_html, max_width=350),
        icon=folium.Icon(
            color=industry_icon_colors.get(industry, "gray"),
            icon=event_type_icons.get(event_type, "info-circle"),
            prefix="fa",
        ),
    ).add_to(m)

m


### Export the map to `maps/mp03_map_team_<NN>.html`

In [23]:
# Export the rendered map to the required path.
#
# Replace TEAM_NUMBER in the setup cell before final export.
maps_dir = repo_root / "maps"
maps_dir.mkdir(exist_ok=True)

OUTPUT_PATH = maps_dir / f"mp03_map_team_{TEAM_NUMBER}.html"
m.save(OUTPUT_PATH)

print(f"Map saved to {OUTPUT_PATH}")


Map saved to /content/maps/mp03_map_team_05.html


MP03 Methodology — Team 05 Industry Comparison: Financial Services and Travel and Hospitality

6.1 Ticker-List Rationale Financial Services:

We started with the seed list and added GS, MS, SCHW, PYPL, and SQ to cover capital markets, brokerage, and fintech firms that weren't in the original list. These companies have physical offices and branches that could show up in location related 8-K filings. Travel and Hospitality: We kept the seed list and added lodging REITs (HST, PK, APLE, etc.), resort/casino operators (MGM, LVS, WYNN), additional airlines (ALK, JBLU, SAVE), and vacation ownership firms (VAC, TNL, HGV). Stage 1 testing showed that a lot of hotel and property related EDGAR hits came from property owners and operators rather than the big brand names like Marriott or Hilton, so expanding the list improved how many relevant filings we found.

6.2 Search-Phrase Rationale Financial Services:

We kept all the seed phrases and added "new office", "office opening", "trading floor", and "advisory office" to catch capital markets and wealth management announcements. We also added company name phrases like "JPMorgan" "branch" and "Goldman Sachs" "office" to target specific firms directly. Even with these additions, EDGAR returned very few candidates because big banks usually don't file standalone 8-Ks for branch changes, they report that kind of thing in earnings releases instead. Travel and Hospitality: We extended the seed phrases with terms like "new resort", "resort expansion", "hotel conversion", "route expansion", and "new destination" to catch more types of location events beyond just hotel openings. We also added some unquoted terms like hotel opening and new hotel to improve recall since EDGAR's search can miss things when phrases are too specific.

6.3 Window-Experiment Results

The results showed:
[industry, window_days, candidate_count, event_count,estimated_cost_usd]
[Financial Services, 30, 0, 0, 0.000000]
[Financial Services, 60, 0, 0, 0.000000]
[Financial Services, 90, 0, 0, 0.000000]
[Financial Services, 180, 0, 0, 0.000000]
[Financial Services, 360, 1, 1, 0.002928]
[Travel and Hospitality, 30, 42, 8, 0.106696]

Travel and Hospitality hit the 8 event target at 30 days and stopped there. Financial Services never reached the target, it returned zero candidates through 180 days and only 1 at 360 days. The chosen window is 360 days because that's the ceiling and Financial Services fell short at every window tested. The shortfall is acknowledged in the limitations section. Total API cost across all trials was about $0.11, which is a good degree under the 3.00 ceiling.

6.4 Stage 3 Classification Quality Travel and Hospitality:

Out of 42 candidates at 30 days, Claude classified 8 as real location events and rejected the other 34. Of the 8, 7 were successfully geocoded. The positive classifications looked very legitimate, they included hotel tower openings, property announcements, and filings relating to resorts with specific cities and states. One event failed geocoding because the city Claude extracted didn't resolve in Nominatim. Financial Services: Only one candidate came through across all windows, and it was classified as a true event but failed geocoding. With only one result there's not much to evaluate.

6.5 Limitations

The biggest limitation is that Financial Services barely showed up in EDGAR at all. Large banks don't usually file standalone 8-Ks for branch openings or closures, they report that in earnings calls or 10-Ks, which our pipeline doesn't search. The ticker filter may also have cut out some valid hits if EDGAR didn't attach ticker metadata to the filing. On the Travel side, one event was lost at geocoding because the city Claude extracted didn't resolve in Nominatim. The 30 day window for Travel also means we're only seeing very recent filings, which might not represent typical patterns over a longer period.

---

## 7. Comparative Reflection

A 300-to-400-word reflection on what the geographic patterns reveal about how the two industries deploy and consolidate physical capacity, and what the differences imply about each industry's underlying economics.

The same content appears as a standalone Markdown file at `reflections/mp03_reflection_team_05.md`.

**Comparative Reflection:**

The map suggests that Financial Services and Travel and Hospitality use physical locations for different business purposes. Financial Services location events are usually tied to efficiency, consolidation, and operational coverage. Banks and related firms often adjust physical capacity through branch changes, office closures, regional office moves, operations centers, or other cost-rationalization decisions. This reflects an industry where digital banking, automation, and centralized back-office work can reduce the need for dense physical networks in some areas.

Travel and Hospitality location events show a different pattern because physical presence is still central to revenue generation. Hotels, resorts, airlines, cruise operators, and vacation ownership companies depend on destinations, properties, routes, airports, and tourism markets. Their location events are more likely to represent expansion, conversion, or demand capture rather than simple cost reduction. A new hotel, resort expansion, route launch, or property conversion usually signals that the company expects customer demand in that geography to support additional capacity.

The contrast matters because the two industries treat geography differently. In Financial Services, geography often reflects where firms can serve customers efficiently while controlling fixed costs. In Travel and Hospitality, geography reflects where customers want to go and where companies believe leisure, business travel, or tourism demand will grow. This means Financial Services events may cluster around offices, branches, and operational hubs, while Travel and Hospitality events may cluster around tourism markets, airports, resort regions, and growing travel destinations.

The comparison has limitations. The results depend on the chosen time window, ticker lists, search phrases, EDGAR filing language, Claude classification, and geocoding accuracy. Some companies may announce location activity outside of 8-K filings, and some filings may mention locations without representing true physical-capacity changes. Still, the mapped results provide a useful starting point for comparing how two industries make location decisions: one shaped more by cost structure and service delivery, the other shaped more by demand growth and destination-based expansion.


---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_<NN>.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_<NN>.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_<NN>.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-<NN>` and submission tag `mp03-team-<NN>` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.